In [22]:
!pip install clingo

zsh:1: command not found: pip


In [23]:
def solve(grid_puzzle: str):
    import clingo
    program = grid_puzzle
    ctl = clingo.Control()
    ctl.add("base", [], program)
    ctl.ground([("base", [])])

    # 0 = enumerate ALL answer sets (default is 1, i.e. just the first)
    ctl.configuration.solve.models = 0

    model_count = 0

    def show_model(model):
        nonlocal model_count
        model_count += 1
        print(f"Answer set {model_count}:")
        print(model.symbols(shown=True))
        print()

    ctl.solve(on_model=show_model)

    print(f"Total solutions found: {model_count}")

In [24]:
print("""
'''
Story:
Match each auctioned spider specimen to its owner and country of origin, and determine the final price paid for each. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.Freddie's purchase cost 15 dollars more than the cart spider.
2.Of the darkback and the cart spider, one sold for $60 and the other was won by Vicky.
3.Of the spider that sold for $75 and Shelley's purchase, one was the archaenido and the other was the cart spider.
4.The arachnid that sold for $90 was either Shelley's purchase or the darkback.

Solve the grid puzzle by filling the table:
+----------+-----------+-----------+
| Prices   | Winners   | Spiders   |
+==========+===========+===========+
| $45      |           |           |
+----------+-----------+-----------+
| $60      |           |           |
+----------+-----------+-----------+
| $75      |           |           |
+----------+-----------+-----------+
| $90      |           |           |
+----------+-----------+-----------+

Use the clues to fill the table with the following categories:
Winners: Charles, Freddie, Shelley, Vicky
Spiders: archaenido, cart spider, darkback, whip spider
""")


'''
Story:
Match each auctioned spider specimen to its owner and country of origin, and determine the final price paid for each. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.Freddie's purchase cost 15 dollars more than the cart spider.
2.Of the darkback and the cart spider, one sold for $60 and the other was won by Vicky.
3.Of the spider that sold for $75 and Shelley's purchase, one was the archaenido and the other was the cart spider.
4.The arachnid that sold for $90 was either Shelley's purchase or the darkback.

Solve the grid puzzle by filling the table:
+----------+-----------+-----------+
| Prices   | Winners   | Spiders   |
+==========+===========+===========+
| $45      |           |           |
+----------+--------

In [25]:
program1 = """
% ============================================================
% Abbeville K-9 Academy grid puzzle - ASP (clingo) encoding
% ============================================================

% ---- Domains ----
months(march;april;may;june).
dogs(aries;barca;hercules;jaws).
officers(daugherty;lyons;quinn;salinas).

% "1 month before" relation over the timeline
order(march,april).
order(april,may).
order(may,june).

% ---- Assignment: each month gets exactly one dog, and vice versa ----
1 { dog(M,D) : dogs(D) } 1 :- months(M).
1 { dog(M,D) : months(M) } 1 :- dogs(D).

% ---- Assignment: each month gets exactly one officer, and vice versa ----
1 { officer(M,O) : officers(O) } 1 :- months(M).
1 { officer(M,O) : months(M) } 1 :- officers(O).

% ============================================================
% Clues
% ============================================================

% Clue 1: The dog that graduated in May was assigned to
%         Officer Salinas or Officer Quinn.
:- officer(may,O), O != salinas, O != quinn.

% Clue 2: Barca graduated in April.
:- dog(april,D), D != barca.

% Clue 3: Officer Lyons's dog graduated exactly 1 month
%         before Officer Daugherty's dog.
:- officer(M1,lyons), officer(M2,daugherty), not order(M1,M2).

% Clue 4: Lyons's dog, the June dog, Jaws, and the April dog
%         are four DIFFERENT dogs (pairwise distinct):
%   - Lyons's dog is not the June dog
:- officer(june,lyons).
%   - Lyons's dog is not the April dog
:- officer(april,lyons).
%   - Jaws is not the June dog
:- dog(june,jaws).
%   - Jaws is not the April dog (also follows from clue 2)
:- dog(april,jaws).
%   - Lyons's dog is not Jaws
:- officer(M,lyons), dog(M,jaws).

% Clue 5: Officer Salinas's dog was Hercules or Barca.
:- officer(M,salinas), dog(M,D), D != hercules, D != barca.

% ============================================================
% Output
% ============================================================
#show dog/2.
#show officer/2.
"""
solve(program1)

Answer set 1:
[dog(march,aries), dog(april,barca), dog(june,hercules), dog(may,jaws), officer(april,daugherty), officer(march,lyons), officer(may,quinn), officer(june,salinas)]

Total solutions found: 1


In [26]:
program2 = """
% --- Domains ---
price(45; 60; 75; 90).
winner(charles; freddie; shelley; vicky).
spider(archaenido; cart_spider; darkback; whip_spider).

% --- Generating Assignments ---
% Each price is paid by exactly one winner, and vice versa.
1 { pays(W, P) : winner(W) } 1 :- price(P).
1 { pays(W, P) : price(P)  } 1 :- winner(W).

% Each price buys exactly one spider, and vice versa.
1 { item(S, P) : spider(S) } 1 :- price(P).
1 { item(S, P) : price(P)  } 1 :- spider(S).

% --- Helper Definitions ---
owner(W, S) :- pays(W, P), item(S, P).

% --- Clue 1 ---
% Freddie's purchase cost 15 dollars more than the cart spider.
:- pays(freddie, PF), item(cart_spider, PC), PF != PC + 15.

% --- Clue 2 ---
% Of the darkback and the cart spider, one sold for $60 and the other was won by Vicky.
:- item(darkback, 60), owner(vicky, darkback).
:- item(cart_spider, 60), owner(vicky, cart_spider).
:- not item(darkback, 60), not owner(vicky, darkback).
:- not item(cart_spider, 60), not owner(vicky, cart_spider).

% --- Clue 3 ---
% Of the spider that sold for $75 and Shelley's purchase, one was the archaenido and the other was the cart spider.
:- item(archaenido, 75), owner(shelley, archaenido).
:- item(cart_spider, 75), owner(shelley, cart_spider).
:- not item(archaenido, 75), not owner(shelley, archaenido).
:- not item(cart_spider, 75), not owner(shelley, cart_spider).

% --- Clue 4 ---
% The arachnid that sold for $90 was either Shelley's purchase or the darkback.
:- item(S, 90), not owner(shelley, S), S != darkback.
:- pays(shelley, 90), item(darkback, 90).

% --- Output Formatting ---
#show pays/2.
#show item/2.
"""
solve(program2)


Answer set 1:
[pays(charles,45), pays(freddie,75), pays(shelley,60), pays(vicky,90), item(archaenido,75), item(cart_spider,60), item(darkback,90), item(whip_spider,45)]

Total solutions found: 1


In [27]:
program2 = """
% ============================================================% Foodie Club logic grid puzzle - ASP (clingo) encoding% ============================================================
% ---- Domains ----%

prices(p24_99; p25_99; p26_99; p27_99).
names(frances; genevieve; lynda; willie).
wines(bordeaux; chianti; merlot; zinfandel).
entrees(beef_stir_fry; filet_mignon; lobster_tail; pork_chops).

% Define price differences for clues% price_diff(P1, P2, D) means P1 is D dollars more than P2

price_diff(p25_99, p24_99, 1).price_diff(p26_99, p25_99, 1).price_diff(p27_99, p26_99, 1).
price_diff(p26_99, p24_99, 2).price_diff(p27_99, p25_99, 2).

% ---- Assignment: each price gets exactly one name, and vice versa ----
1 { name(P,N) : names(N) } 1 :- prices(P).
1 { name(P,N) : prices(P) } 1 :- names(N).

% ---- Assignment: each price gets exactly one wine, and vice versa ----
1 { wine(P,W) : wines(W) } 1 :- prices(P).
1 { wine(P,W) : prices(P) } 1 :- wines(W).

% ---- Assignment: each price gets exactly one entree, and vice versa ----
1 { entree(P,E) : entrees(E) } 1 :- prices(P).
1 { entree(P,E) : prices(P) } 1 :- entrees(E).

% ============================================================% Clues% ============================================================
% Clue 1: Lynda paid 2 dollars more than the diner who ordered the beef stir-fry.
:- name(P1, lynda), entree(P2, beef_stir_fry), not price_diff(P1, P2, 2).

% Clue 2: The diner who ordered the beef stir-fry was either the one who had the merlot or the one who had the bordeaux.
:- entree(P, beef_stir_fry), not wine(P, merlot), not wine(P, bordeaux).

% Clue 3: The person who paid $26.99 didn't have the zinfandel.
:- wine(p26_99, zinfandel).

% Clue 4: The one who had the chianti paid 2 dollars less than the diner who ordered the filet mignon.% This means the filet mignon diner paid 2 dollars more than the chianti diner.
:- wine(P1, chianti), entree(P2, filet_mignon), not price_diff(P2, P1, 2).

% Clue 5: Of the one who had the merlot and the one who had the chianti,% one paid $25.99 and the other ordered the lobster tail.
% Helper predicates for the two possibilities
clue5_possibility_1(P_merlot, P_chianti) :- P_merlot = p25_99, entree(P_chianti, lobster_tail).
clue5_possibility_2(P_merlot, P_chianti) :- P_chianti = p25_99, entree(P_merlot, lobster_tail).

% Constraint: one of the possibilities must be true
:- wine(P_merlot, merlot), wine(P_chianti, chianti), P_merlot != P_chianti,
not clue5_possibility_1(P_merlot, P_chianti),
not clue5_possibility_2(P_merlot, P_chianti).

% Clue 6: The diner who ordered the beef stir-fry paid 1 dollar more than Frances.
:- entree(P1, beef_stir_fry), name(P2, frances), not price_diff(P1, P2, 1).

% Clue 7: Neither Willie nor the diner who ordered the beef stir-fry was the person who paid $27.99.
:- name(p27_99, willie).
:- entree(p27_99, beef_stir_fry).
:- name(P, willie), entree(P, beef_stir_fry).
% ============================================================% Output% ============================================================#
#show name/2.
#show wine/2.
#show entree/2.
"""
solve(program2)

Answer set 1:
[name(p24_99,frances), name(p25_99,genevieve), name(p27_99,lynda), name(p26_99,willie), wine(p26_99,bordeaux), wine(p24_99,chianti), wine(p25_99,merlot), wine(p27_99,zinfandel), entree(p25_99,beef_stir_fry), entree(p26_99,filet_mignon), entree(p24_99,lobster_tail), entree(p27_99,pork_chops)]

Total solutions found: 1


# Another Puzzle

In [28]:
!pip install clingo

zsh:1: command not found: pip


In [29]:
def solve(grid_puzzle: str):
    import clingo
    program = grid_puzzle
    ctl = clingo.Control()
    ctl.add("base", [], program)
    ctl.ground([("base", [])])

    # 0 = enumerate ALL answer sets (default is 1, i.e. just the first)
    ctl.configuration.solve.models = 0

    model_count = 0

    def show_model(model):
        nonlocal model_count
        model_count += 1
        print(f"Answer set {model_count}:")
        print(model.symbols(shown=True))
        print()

    ctl.solve(on_model=show_model)

    print(f"Total solutions found: {model_count}")

In [30]:
print('''
Story:
Bushnell County is putting together a historical brochure about some of the ancient cultures that used to live there.  Match each of the ancient cultures to their time period, location, and estimated population size. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The culture located near Waterloo was either the culture with 5,000 people or the Juad culture.
2.The culture located near Unity didn't have a population size of 9,000.
3.The culture founded in 600 AD was either the culture with 4,500 people or the Juad culture.
4.Neither the Budel culture nor the society founded in 100 AD was the culture located near Unity.
5.The culture with 5,000 people was either the culture founded in 350 AD or the culture located near Junction City.
6.The culture with 8,000 people was founded 250 years before the culture with 5,000 people.
7.The Juad culture was either the culture located near Junction City or the society founded in 600 AD.
8.Of the culture located near Quasqueton and the culture located near Waterloo, one had a population size of 5,000 and the other was the Ryat culture.
9.The Aldeton culture was founded sometime after the culture located near Junction City.

Solve the grid puzzle by filling the table:
+---------+------------+---------------+-------------+
| Years   | Cultures   | Populations   | Locations   |
+=========+============+===============+=============+
| 100 AD  |            |               |             |
+---------+------------+---------------+-------------+
| 350 AD  |            |               |             |
+---------+------------+---------------+-------------+
| 600 AD  |            |               |             |
+---------+------------+---------------+-------------+
| 850 AD  |            |               |             |
+---------+------------+---------------+-------------+

Use the clues to fill the table with the following categories:
Cultures: Aldeton, Budel, Juad, Ryat
Populations: 4,500, 5,000, 8,000, 9,000
Locations: Junction City, Quasqueton, Unity, Waterloo
''')


Story:
Bushnell County is putting together a historical brochure about some of the ancient cultures that used to live there.  Match each of the ancient cultures to their time period, location, and estimated population size. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The culture located near Waterloo was either the culture with 5,000 people or the Juad culture.
2.The culture located near Unity didn't have a population size of 9,000.
3.The culture founded in 600 AD was either the culture with 4,500 people or the Juad culture.
4.Neither the Budel culture nor the society founded in 100 AD was the culture located near Unity.
5.The culture with 5,000 people was either the culture founded in 350 AD or the culture located near Ju

In [31]:
program3 = """
% ---- Domains ----
years(100; 350; 600; 850).
cultures(aldeton; budel; juad; ryat).
populations(4500; 5000; 8000; 9000).
locations(junction_city; quasqueton; unity; waterloo).

% ---- Assignment ----
1 {culture(C,Y): cultures(C)} 1 :- years(Y).
1 {culture(C,Y): years(Y)} 1 :- cultures(C).

1 {population(P,Y): populations(P)} 1 :- years(Y).
1 {population(P,Y): years(Y)} 1 :- populations(P).

1 {location(L,Y): locations(L)} 1 :- years(Y).
1 {location(L,Y): years(Y)} 1 :- locations(L).

% ---- Clues ----
% Clue 1: The culture located near Waterloo was either the culture with 5,000 people or the Juad culture.
:- location(waterloo, Y), not culture(juad, Y), not population(5000, Y).

% Clue 2: The culture located near Unity didn't have a population size of 9,000.
:- location(unity, Y), population(9000, Y).

% Clue 3: The culture founded in 600 AD was either the culture with 4,500 people or the Juad culture.
o0 :- culture(C, 600), population(4500, 600).
o1 :- culture(C, 600), C == juad.
:- not 1 {o0; o1} 1.

% Clue 4: Neither the Budel culture nor the society founded in 100 AD was the culture located near Unity.
:- culture(budel, Y), location(unity, Y).
:- location(unity, 100).
:- culture(budel, 100).

% Clue 5: The culture with 5,000 people was either the culture founded in 350 AD or the culture located near Junction City.
:- population(5000, Y), Y != 350, not location(junction_city, Y).

% Clue 6: The culture with 8,000 people was founded 250 years before the culture with 5,000 people.
:- population(8000, Y), not population(5000, Y + 250).

% Clue 7: The Juad culture was either the culture located near Junction City or the society founded in 600 AD.
o2 :- culture(juad, Y), Y == 600.
o3 :- culture(juad, Y), location(junction_city, Y).
:- not 1 {o2; o3} 1.

% Clue 8: Of the culture located near Quasqueton and the culture located near Waterloo, one had a population size of 5,000 and the other was the Ryat culture.
o4 :-
    location(quasqueton, Y), population(5000, Y),
    location(waterloo, Z), culture(ryat, Z).
o5 :-
    location(quasqueton, Y), culture(ryat, Y),
    location(waterloo, Z), population(5000, Z).
:- not 1 {o4; o5} 1.

% Clue 9: The Aldeton culture was founded sometime after the culture located near Junction City.
:- culture(aldeton, Y), location(junction_city, Z), Y <= Z.

% ---- Output ----
#show culture/2.
#show population/2.
#show location/2.
"""
solve(program3)

Answer set 1:
[culture(aldeton,600), culture(budel,350), culture(juad,100), culture(ryat,850), population(4500,600), population(5000,350), population(8000,100), population(9000,850), location(junction_city,100), location(quasqueton,850), location(unity,600), location(waterloo,350)]

Total solutions found: 1


In [32]:
story = '''
Story:
The Windhaven Beach Kite Festival transformed the coastline into a canvas of color as handcrafted kites danced against the blue sky. Festival judge Lily Chen evaluated each entry for design, flight performance, and craftsmanship while spectators spread blankets on the sand to watch the aerial display. Determine the wingspan of each kite, its design style, the material it's made from, and the builder who crafted it. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The silk craft is 2 ft narrower than the rokkaku kite.
2.The nylon craft is somewhat wider than the silk craft.
3.The nylon kite is 2 ft narrower than the ripstop craft.
4.Of the sled craft and the box kite, one is made of nylon and the other has a 3 ft wingspan.
5.The parafoil kite isn't made of silk.
6.The fiberglass kite was either the sled craft or the 4 ft kite.

Solve the grid puzzle by filling the table:
+-------------+-----------+-------------+
| Wingspans   | Designs   | Materials   |
+=============+===========+=============+
| 3 ft        |           |             |
+-------------+-----------+-------------+
| 4 ft        |           |             |
+-------------+-----------+-------------+
| 5 ft        |           |             |
+-------------+-----------+-------------+
| 6 ft        |           |             |
+-------------+-----------+-------------+
| 7 ft        |           |             |
+-------------+-----------+-------------+

Use the clues to fill the table with the following categories:
Designs: box, diamond, parafoil, rokkaku, sled
Materials: bamboo, fiberglass, nylon, ripstop, silk
'''
print(story)


Story:
The Windhaven Beach Kite Festival transformed the coastline into a canvas of color as handcrafted kites danced against the blue sky. Festival judge Lily Chen evaluated each entry for design, flight performance, and craftsmanship while spectators spread blankets on the sand to watch the aerial display. Determine the wingspan of each kite, its design style, the material it's made from, and the builder who crafted it. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The silk craft is 2 ft narrower than the rokkaku kite.
2.The nylon craft is somewhat wider than the silk craft.
3.The nylon kite is 2 ft narrower than the ripstop craft.
4.Of the sled craft and the box kite, one is made of nylon and the other has a 3 ft wingspan

In [33]:
program4 = """
% ---- Domains ----
wingspans(3; 4; 5; 6; 7).
designs(box; diamond; parafoil; rokkaku; sled).
materials(bamboo; fiberglass; nylon; ripstop; silk).

% ---- Assignment ----
1 {design(D,W): designs(D)} 1 :- wingspans(W).
1 {design(D,W): wingspans(W)} 1 :- designs(D).

1 {material(M,W): materials(M)} 1 :- wingspans(W).
1 {material(M,W): wingspans(W)} 1 :- materials(M).

% ---- Clues ----
% Clue 1: The silk craft is 2 ft narrower than the rokkaku kite.
o1 :-material(silk, W), design(rokkaku, W + 2).
:- not o1.

% Clue 2: The nylon craft is somewhat wider than the silk craft.
o2 :- material(nylon, W1), material(silk, W2), W1 > W2.
:- not o2.

% Clue 3: The nylon kite is 2 ft narrower than the ripstop craft.
o3 :- material(nylon, W), material(ripstop, W + 2).
:- not o3.

% Clue 4: Of the sled craft and the box kite, one is made of nylon and the other has a 3 ft wingspan.
o4_1 :- design(sled, W1), material(nylon, W1), design(box, W2), wingspans(W2), W2 = 3.
o4_2 :- design(sled, W1), wingspans(W1), W1 = 3, design(box, W2), material(nylon, W2).
:- not 1 {o4_1; o4_2} 1.

% Clue 5: The parafoil kite isn't made of silk.
:- design(parafoil, W), material(silk, W).

% Clue 6: The fiberglass kite was either the sled craft or the 4 ft kite.
o5_1 :- material(fiberglass, W), design(sled, W).
o5_2 :- material(fiberglass, W), wingspans(W), W = 4.
:- not 1 {o5_1; o5_2} 1.

% ---- Output ----
#show material/2.
#show design/2.
"""
solve(program4)

Answer set 1:
[design(box,5), design(diamond,4), design(parafoil,7), design(rokkaku,6), design(sled,3), material(bamboo,6), material(fiberglass,3), material(nylon,5), material(ripstop,7), material(silk,4)]

Total solutions found: 1


In [34]:
print('''
In the state of Vermont, a team is compiling an educational guide about some of the indigenous groups that once inhabited the region. Match each indigenous group to their time period, location, and estimated population size. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once. If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The group located near Lake Champlain was either the group with 6,000 people or the Wabanaki culture.
2.The group located near Burlington didn't have a population size of 7,000.
3.The group founded in 1500 AD was either the group with 4,000 people or the Wabanaki culture.
4.Neither the Abenaki culture nor the society founded in 800 AD was the group located near Burlington.
5.The group with 6,000 people was either the group founded in 1200 AD or the group located near Montpelier.
6.The group with 9,000 people was founded 300 years before the group with 6,000 people.
7.The Wabanaki culture was either the group located near Montpelier or the society founded in 1500 AD.
8.Of the group located near Rutland and the group located near Lake Champlain, one had a population size of 6,000 and the other was the Mohican culture.
9.The Abenaki culture was founded sometime after the group located near Montpelier.

Solve the grid puzzle by filling the table:
+---------+------------+---------------+-------------+
| Years   | Cultures   | Populations   | Locations   |
+=========+============+===============+=============+
| 800 AD  |            |               |             |
+---------+------------+---------------+-------------+
| 1200 AD |            |               |             |
+---------+------------+---------------+-------------+
| 1500 AD |            |               |             |
+---------+------------+---------------+-------------+
| 1800 AD |            |               |             |
+---------+------------+---------------+-------------+

Use the clues to fill the table with the following categories:
Cultures: Abenaki, Mohican, Wabanaki, Algonquin
Populations: 4,000, 6,000, 9,000, 7,000
Locations: Montpelier, Rutland, Burlington, Lake Champlain
''')


In the state of Vermont, a team is compiling an educational guide about some of the indigenous groups that once inhabited the region. Match each indigenous group to their time period, location, and estimated population size. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once. If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.The group located near Lake Champlain was either the group with 6,000 people or the Wabanaki culture.
2.The group located near Burlington didn't have a population size of 7,000.
3.The group founded in 1500 AD was either the group with 4,000 people or the Wabanaki culture.
4.Neither the Abenaki culture nor the society founded in 800 AD was the group located near Burlington.
5.The group with 6,000 people was either the group founded in 1200 AD or the group locat

In [35]:
program5 = """
% ---- Domains ----
years(800; 1200; 1500; 1800).
cultures(abenaki; mohican; wabanaki; algonquin).
locations(montpelier; rutland; burlington; lake_champlain).
populations(4000; 6000; 9000; 7000).

% ---- Assignment ----
1 {culture(C,Y): cultures(C)} 1 :- years(Y).
1 {culture(C,Y): years(Y)} 1 :- cultures(C).

1 {location(L,Y): locations(L)} 1 :- years(Y).
1 {location(L,Y): years(Y)} 1 :- locations(L).

1 {population(P,Y): populations(P)} 1 :- years(Y).
1 {population(P,Y): years(Y)} 1 :- populations(P).

% ---- Clues ----
% Clue: 1: The group located near Lake Champlain was either the group with 6,000 people or the Wabanaki culture.
o1_1 :- location(lake_champlain, Y), population(6000, Y).
o1_2 :- location(lake_champlain, Y), culture(wabanaki, Y).
:- not 1 {o1_1; o1_2} 1.
    
% Clue: 2: The group located near Burlington didn't have a population size of 7,000.
:- location(burlington, Y), population(7000, Y).

% Clue: 3: The group founded in 1500 AD was either the group with 4,000 people or the Wabanaki culture.
o3_1 :- years(Y), population(4000, Y), Y = 1500.
o3_2 :- years(Y), culture(wabanaki, Y), Y = 1500.
:- not 1 {o3_1; o3_2} 1.

% Clue: 4: Neither the Abenaki culture nor the society founded in 800 AD was the group located near Burlington.
o4_1 :- culture(abenaki, Y), location(burlington, Y).
o4_2 :- years(Y), location(burlington, Y), Y = 800.
:- o4_1.
:- o4_2.

% Clue: 5: The group with 6,000 people was either the group founded in 1200 AD or the group located near Montpelier.
o5_1 :- population(6000, Y), years(1200), Y = 1200.
o5_2 :- population(6000, Y), location(montpelier, Y).
:- not 1 {o5_1; o5_2} 1.

% Clue: 6: The group with 9,000 people was founded 300 years before the group with 6,000 people.
o6 :- population(9000, Y1), population(6000, Y2), Y2 = Y1 + 300.
:- not o6.

% Clue: 7: The Wabanaki culture was either the group located near Montpelier or the society founded in 1500 AD.
o7_1 :- culture(wabanaki, Y), location(montpelier, Y).
o7_2 :- culture(wabanaki, Y), years(1500), Y = 1500.
:- not 1 {o7_1; o7_2} 1.

% Clue: 8: Of the group located near Rutland and the group located near Lake Champlain, one had a population size of 6,000 and the other was the Mohican culture.
o8_1 :- location(rutland, Y), population(6000, Y), location(lake_champlain, Y2), culture(mohican, Y2).
o8_2 :- location(rutland, Y), culture(mohican, Y), location(lake_champlain, Y2), population(6000, Y2).
:- not 1 {o8_1; o8_2} 1.

% Clue: 9: The Abenaki culture was founded sometime after the group located near Montpelier.
:- culture(abenaki, Y1), location(montpelier, Y2), Y1 <= Y2.

% ---- Output ----
#show location/2.
#show culture/2.
#show population/2.
"""
solve(program5)

Total solutions found: 0


In [36]:
print(
'''
At the Evergreen Art Museum, a curator named Lucas is cataloging a unique collection of paper sculptures — from intricate butterflies to towering giraffes, each crafted from different types of paper. Match each sculpture to its material, theme, and creator, as well as determine the number of folds used in its creation. Remember, no option in any category will be repeated. If you encounter difficulties, try using the "Clear Errors" or "Hint" features to assist with solving.

Clues:
1.The sculpture made from recycled paper has 25 folds.
2.Of the sculptures depicting a giraffe and the piece with 35 folds, one is crafted from hemp and the other belongs to Elena.
3.Elena's sculpture has 5 more folds than Daniel's artwork.
4.The sculpture representing a butterfly has 15 fewer folds than Olivia's piece.
5.The sculpture made from handmade paper has 20 more folds than the creation made from recycled paper.
6.Daniel's piece has 5 less folds than the sculpture depicting a cat.
7.The sculpture with 40 folds, the one made from hemp, Elena's artwork and James's sculpture are all distinct pieces.
8.Olivia's creation depicts a whale.
9.The sculpture made from kraft paper isn't Olivia's.
10.Lucas's sculpture has more than James's.

Solve the grid puzzle by filling the table:
+---------------+-----------------+------------+----------+
| Fold Counts   | Paper Types     | Subjects   | People   |
+===============+=================+============+==========+
| 25 folds      |                 |            |          |
+---------------+-----------------+------------+----------+
| 30 folds      |                 |            |          |
+---------------+-----------------+------------+----------+
| 35 folds      |                 |            |          |
+---------------+-----------------+------------+----------+
| 40 folds      |                 |            |          |
+---------------+-----------------+------------+----------+
| 45 folds      |                 |            |          |
+---------------+-----------------+------------+----------+

Use the clues to fill the table with the following categories:
Paper Types: recycled, hemp, handmade, kraft, washi
Subjects: butterfly, giraffe, flower, cat, whale
People: Daniel, Elena, James, Olivia, Lucas
'''
)


At the Evergreen Art Museum, a curator named Lucas is cataloging a unique collection of paper sculptures — from intricate butterflies to towering giraffes, each crafted from different types of paper. Match each sculpture to its material, theme, and creator, as well as determine the number of folds used in its creation. Remember, no option in any category will be repeated. If you encounter difficulties, try using the "Clear Errors" or "Hint" features to assist with solving.

Clues:
1.The sculpture made from recycled paper has 25 folds.
2.Of the sculptures depicting a giraffe and the piece with 35 folds, one is crafted from hemp and the other belongs to Elena.
3.Elena's sculpture has 5 more folds than Daniel's artwork.
4.The sculpture representing a butterfly has 15 fewer folds than Olivia's piece.
5.The sculpture made from handmade paper has 20 more folds than the creation made from recycled paper.
6.Daniel's piece has 5 less folds than the sculpture depicting a cat.
7.The sculpture wi

In [37]:
program6 = """
% ---- Domains ----
fold_counts(25; 30; 35; 40; 45).
paper_types(recycled; hemp; handmade; kraft; washi).
subjects(butterfly; giraffe; flower; cat; whale).
peoples(daniel; elena; james; olivia; lucas).

% ---- Assignment ----
1 {paper_type(P, Y): paper_types(P)} 1 :- fold_counts(Y).
1 {paper_type(P, Y): fold_counts(Y)} 1 :- paper_types(P).

1 {subject(S, Y): subjects(S)} 1 :- fold_counts(Y).
1 {subject(S, Y): fold_counts(Y)} 1 :- subjects(S).

1 {people(P, Y): peoples(P)} 1 :- fold_counts(Y).
1 {people(P, Y): fold_counts(Y)} 1 :- peoples(P).

% ---- Clues ----
%Clues 1: The sculpture made from recycled paper has 25 folds.
o1 :- paper_type(recycled, F), F = 25.
:- not o1.

%Clues 2: Of the sculptures depicting a giraffe and the piece with 35 folds, one is crafted from hemp and the other belongs to Elena.
o2_1 :- subject(giraffe, F1), fold_counts(F2), paper_type(hemp, F1), people(elena, F2), F1 != F2, F2 = 35.
o2_2 :- subject(giraffe, F1), fold_counts(F2), paper_type(hemp, F2), people(elena, F1), F1 != F2, F1 = 35.
:- not 1 {o2_1; o2_2} 1.

%Clues 3: Elena's sculpture has 5 more folds than Daniel's artwork.
o3 :- people(elena, F1), people(daniel, F2), F1 = F2 + 5.
:- not o3.


%Clues 4: The sculpture representing a butterfly has 15 fewer folds than Olivia's piece.
o4 :- subject(butterfly, F1), people(olivia, F2), F1 = F2 - 15.
:- not o4.

%Clues 5: The sculpture made from handmade paper has 20 more folds than the creation made from recycled paper.
o5 :- paper_type(handmade, F1), paper_type(recycled, F2), F1 = F2 + 20.
:- not o5.

%Clues 6: Daniel's piece has 5 less folds than the sculpture depicting a cat.
o6 :- people(daniel, F1), subject(cat, F2), F1 = F2 - 5.
:- not o6.


%Clues 7: The sculpture with 40 folds, the one made from hemp, Elena's artwork and James's sculpture are all distinct pieces.
o7 :- fold_counts(40), paper_type(hemp, F1), people(elena, F2), people(james, F3), F1 != 40, F2 != 40, F3 != 40, F1 != F2, F1 != F3, F2 != F3.
:- not o7.

%Clues 8: Olivia's creation depicts a whale.
o8 :- people(olivia, F), subject(whale, F).
:- not o8.

%Clues 9: The sculpture made from kraft paper isn't James's.
o9 :- paper_type(kraft, F1), people(olivia, F2), F1 != F2.
:- not o9.

%Clues 10: Lucas's sculpture has more than James's.
o10 :- people(lucas, F1), people(james, F2), F1 > F2.
:- not o10.

% ---- Output ----
#show paper_type/2.
#show subject/2.
#show people/2.
"""
solve(program6)

Answer set 1:
[paper_type(recycled,25), paper_type(hemp,30), paper_type(handmade,45), paper_type(kraft,35), paper_type(washi,40), subject(butterfly,25), subject(giraffe,30), subject(flower,45), subject(cat,35), subject(whale,40), people(daniel,30), people(elena,35), people(james,25), people(olivia,40), people(lucas,45)]

Total solutions found: 1


In [38]:
print(
'''
Story:
Miravella is writing an article about some of the richest people in the world.  Using only the clues that follow, match each person to his or her country of origin, and determine their total wealth (in dollars) and the type of business they're involved in. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.Gil Gallegos is worth 2 billion less than Nadine Newton.
2.Of the tycoon worth $25 billion and Gil Gallegos, one is from France and the other is from Mexico.
3.Ida Ingram is from Germany.
4.Joseph Jakes is from Mexico.
5.The person in Germany has more wealth than the person in Mexico.

Solve the grid puzzle by filling the table:
+-------------+----------+-------------+
| Wealth      | People   | Countries   |
+=============+==========+=============+
| $25 billion |          |             |
+-------------+----------+-------------+
| $26 billion |          |             |
+-------------+----------+-------------+
| $27 billion |          |             |
+-------------+----------+-------------+
| $28 billion |          |             |
+-------------+----------+-------------+

Use the clues to fill the table with the following categories:
People: Gil Gallegos, Ida Ingram, Joseph Jakes, Nadine Newton
Countries: Ireland, France, Germany, Mexico
'''
)


Story:
Miravella is writing an article about some of the richest people in the world.  Using only the clues that follow, match each person to his or her country of origin, and determine their total wealth (in dollars) and the type of business they're involved in. Remember, as with all grid-based logic puzzles, no option in any category will ever be used more than once.  If you get stuck or run into problems, try the "Clear Errors" button to remove any mistakes that might be present on the grid, or the "Hint" button to see the next logical step in the puzzle.

Clues:
1.Gil Gallegos is worth 2 billion less than Nadine Newton.
2.Of the tycoon worth $25 billion and Gil Gallegos, one is from France and the other is from Mexico.
3.Ida Ingram is from Germany.
4.Joseph Jakes is from Mexico.
5.The person in Germany has more wealth than the person in Mexico.

Solve the grid puzzle by filling the table:
+-------------+----------+-------------+
| Wealth      | People   | Countries   |
+==========

In [47]:
program7 = """
% ---- Domains ----
wealths(25; 26; 27; 28).
peoples(gil; ida; joseph; nadine).
countries(ireland; france; germany; mexico).

% ---- Assignment ----
1 {people(P, W): peoples(P)} 1 :- wealths(W).
1 {people(P, W): wealths(W)} 1 :- peoples(P).

1 {country(C, W): countries(C)} 1 :- wealths(W).
1 {country(C, W): wealths(W)} 1 :- countries(C).

% ---- Clues ----
%Clue 1: Gil Gallegos is worth 2 billion less than Nadine Newton.
o1 :- people(gil, W1), people(nadine, W2), W1 + 2 == W2.
:- not o1.

%Clue 2: Of the tycoon worth $25 billion and Gil Gallegos, one is from France and the other is from Mexico.
o2_1 :- people(gil, W2), W1 = 25, W1 != W2, country(france, W1), country(mexico, W2).
o2_2 :- people(gil, W2), W1 = 25, W1 != W2, country(mexico, W1), country(france, W2).
:- not 1 {o2_1; o2_2} 1.

%Clue 3: Ida Ingram is from Germany.
o3 :- people(ida, W), country(germany, W).
:- not o3.

%Clue 4: Joseph Jakes is from Mexico.
o4 :- people(joseph, W), country(mexico, W).
:- not o4.

%Clue 5: The person in Germany has more wealth than the person in Mexico.
o5 :- country(germany, W1), country(mexico, W2), W1 > W2.
:- not o5.

% ---- Output ----
#show people/2.
#show country/2.
"""
solve(program7)

Answer set 1:
[people(gil,26), people(ida,27), people(joseph,25), people(nadine,28), country(ireland,28), country(france,26), country(germany,27), country(mexico,25)]

Total solutions found: 1
